# HW_L03_03_ML

## 1. Initial Plan and Preprocessing

In this exercise, you will use the Scikit-Learn library to build a complete machine learning pipeline. You will go beyond simple data analysis and build predictive models. The main focus is on data splitting, feature engineering, model selection, and hyperparameter tuning. In this exercise, you will use a synthetic dataset (`delivery_data.csv`) that simulates data from a shipping company, and you must train two machine learning models:

1. Regression model: To accurately estimate delivery time in minutes for display to the customer.
2. Classification model: To predict whether an order will be delivered with a delay (more than 45 minutes) or not (to send a discount code or a proactive apology).


In [2]:
# basic
import os

# Analysis
import numpy as np
import pandas as pd

# plot
import matplotlib.pyplot as plt
import seaborn as sns

## Part 1
Before training the model, we must prepare the data. For this purpose, the following steps must be followed:

1. Loading and Splitting:
   - Load the dataset.
   - Define the feature matrix (X) and the target vector (y). (Currently, use the `Delivery_Time_min` column for the target.)
   - Using `train_test_split`, split the data into training (80%) and testing (20%) sets. Be sure to use `random_state` so that results are reproducible.

2. Building the Transformation Pipeline

- We have two types of features: numerical features (`Distance_km`, etc.) and categorical features (`Traffic_Level`, `Weather`).
- Build a `ColumnTransformer` that performs the following operations:
  - For numerical columns: fill missing values with the mean (`SimpleImputer`) and then standardize the data (`StandardScaler`).
  - For categorical columns: fill missing values with the most frequent value (`most_frequent`) and then convert text into vectors (`OneHotEncoder`).
- Note: Linear models require the data to be scaled and cannot process string values directly.



In [4]:
# Read data
delivery_time_min = pd.read_csv("delivery_data.csv")

In [6]:
delivery_time_min.columns

Index(['Distance_km', 'Prep_Time_min', 'Courier_Experience_yrs', 'Weather',
       'Traffic_Level', 'Vehicle_Type', 'Delivery_Time_min', 'Is_Late'],
      dtype='str')

In [7]:
delivery_time_min.head()

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Weather,Traffic_Level,Vehicle_Type,Delivery_Time_min,Is_Late
0,5.993428,11.624109,7,NaN,High,Scooter,44.013542,0
1,4.723471,14.277407,2,Sunny,High,Car,35.527712,0
2,6.295377,11.037900,9,Rainy,High,Scooter,48.567934,1
3,8.046060,13.460192,7,Sunny,High,Car,50.452710,1
4,4.531693,5.531927,7,Rainy,Medium,Bicycle,47.632817,1


In [8]:
delivery_time_min.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Distance_km             2000 non-null   float64
 1   Prep_Time_min           1895 non-null   float64
 2   Courier_Experience_yrs  2000 non-null   int64  
 3   Weather                 1893 non-null   str    
 4   Traffic_Level           2000 non-null   str    
 5   Vehicle_Type            2000 non-null   str    
 6   Delivery_Time_min       2000 non-null   float64
 7   Is_Late                 2000 non-null   int64  
dtypes: float64(3), int64(2), str(3)
memory usage: 125.1 KB


In [9]:
delivery_time_min.describe()

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Delivery_Time_min,Is_Late
count,2000.000000,1895.000000,2000.000000,2000.000000,2000.000000
mean,5.097968,15.031132,4.552500,53.317833,0.549000
std,1.956540,4.942245,2.885898,27.200221,0.497718
min,0.500000,5.000000,0.000000,10.000000,0.000000
25%,3.754676,11.453845,2.000000,33.683481,0.000000
50%,5.089383,15.033999,5.000000,47.682737,1.000000
75%,6.365955,18.350682,7.000000,67.152951,1.000000
max,12.705463,34.631189,9.000000,183.617375,1.000000


In [10]:
delivery_time_min.isnull().sum()

Distance_km                 0
Prep_Time_min             105
Courier_Experience_yrs      0
Weather                   107
Traffic_Level               0
Vehicle_Type                0
Delivery_Time_min           0
Is_Late                     0
dtype: int64

The two columns `Prep_Time_min` and `Weather` contain NaN values, so we fill them with the mean (`SimpleImputer`) and then standardize the data (`StandardScaler`).


In [20]:
delivery_time_min.drop(columns=["Is_Late"]).copy()

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Weather,Traffic_Level,Vehicle_Type,Delivery_Time_min
0,5.993428,11.624109,7,NaN,High,Scooter,44.013542
1,4.723471,14.277407,2,Sunny,High,Car,35.527712
2,6.295377,11.037900,9,Rainy,High,Scooter,48.567934
3,8.046060,13.460192,7,Sunny,High,Car,50.452710
4,4.531693,5.531927,7,Rainy,Medium,Bicycle,47.632817
...,...,...,...,...,...,...,...
1995,7.140300,15.142288,9,Sunny,High,Bicycle,75.866816
1996,4.946957,5.000000,7,Foggy,Low,Car,20.494360
1997,3.236251,13.398511,7,Snowy,High,Bicycle,74.190882
1998,4.673866,23.216891,8,Sunny,Medium,Bicycle,62.662841


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


# drop Is_Late columns 
data_frame_dalivery = delivery_time_min.drop(columns=["Is_Late"]).copy()

# 1st part: Create x and y
# label columns or target
target = "Delivery_Time_min"

# create X and y
# X: model learn - X is matrix so X is Capital charechter
X = data_frame_dalivery.drop(columns=[target])
# y: Answer or label - y is vector and Small charechter
y = data_frame_dalivery[target]


# 2nd part: train_test_split
# user train_test_split for creat train data and test data
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.2,
                                                    random_state = 1)

# 3rd part: ColumnTransformer
# numeric columns and categorical columns
numerical_columns = ["Distance_km", "Prep_Time_min", "Courier_Experience_yrs"]
categorical_columns = ["Weather", "Traffic_Level", "Vehicle_Type"] # this is str

# pipline for numerical columns
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="mean")),
                                  ("scaler", StandardScaler())])

categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("one_hot", OneHotEncoder(handle_unknown="ignore"))])


preprocessor = ColumnTransformer(
    transformers=[("num", numerical_transformer, numerical_columns),
                  ("cat", categorical_transformer, categorical_columns)])

test data by preprocessor

In [ ]:
len((delivery_time_min["Weather"].unique().dropna())

<StringArray>
['Sunny', 'Rainy', 'Snowy', 'Foggy']
Length: 4, dtype: str

In [37]:
delivery_time_min.head(2)

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Weather,Traffic_Level,Vehicle_Type,Delivery_Time_min,Is_Late
0,5.993428,11.624109,7,NaN,High,Scooter,44.013542,0
1,4.723471,14.277407,2,Sunny,High,Car,35.527712,0


In [36]:
pd.DataFrame(preprocessor.fit_transform(X_train))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.450928,-0.713998,0.854953,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.216522,0.477746,-1.221856,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.382447,0.594076,-0.875721,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,1.508346,-1.411686,-1.567990,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,0.432828,-0.062792,-0.183451,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,0.061359,-0.443400,-0.875721,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1596,0.024217,1.054088,-0.875721,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1597,-0.530961,0.895269,1.547222,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1598,0.590966,0.733857,0.508818,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# 1st part
# label columns or target
target = "Delivery_Time_min"

# create X and y
# X: model learn - X is matrix so X is Capital charechter
X = delivery_time_min.drop(columns=[target])
# y: Answer or label - y is vector and Small charechter
y = delivery_time_min[target]

# 2nd part
# user train_test_split for creat train data and test data
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.20,
                                                    random_state=0)

# 3rd part
# numeric columns and categorical columns
numeric_columns = ["Distance_km", "Prep_Time_min", "Courier_Experience_yrs"]
categorical_columns = ["Weather", "Traffic_Level", "Vehicle_Type"] # this is str

# 4th part
numeric_transformer = make_pipeline(SimpleImputer(strategy="mean"),
                                    StandardScaler())

categorical_transformer = make_pipeline(SimpleImputer(strategy="most_frequent"),
                                        OneHotEncoder())

# 5th part
# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_columns), # (name, transformer, columns)
                  ("cat", categorical_transformer, categorical_columns),])

# 6th part
model = make_pipeline(preprocessor, LinearRegression())

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [17]:
delivery_time_min

,Distance_km,Prep_Time_min,Courier_Experience_yrs,Weather,Traffic_Level,Vehicle_Type,Delivery_Time_min,Is_Late
0,5.993428,11.624109,7,NaN,High,Scooter,44.013542,0
1,4.723471,14.277407,2,Sunny,High,Car,35.527712,0
2,6.295377,11.037900,9,Rainy,High,Scooter,48.567934,1
3,8.046060,13.460192,7,Sunny,High,Car,50.452710,1
4,4.531693,5.531927,7,Rainy,Medium,Bicycle,47.632817,1
...,...,...,...,...,...,...,...,...
1995,7.140300,15.142288,9,Sunny,High,Bicycle,75.866816,1
1996,4.946957,5.000000,7,Foggy,Low,Car,20.494360,0
1997,3.236251,13.398511,7,Snowy,High,Bicycle,74.190882,1
1998,4.673866,23.216891,8,Sunny,Medium,Bicycle,62.662841,1
